# **Stock Market Data Cleaning, EDA and Statistics**

studies **AAPL, MSFT and SPY** using approximately five years of daily stock-market data.

The work is divided into three simple parts:
1. Data cleaning
2. Exploratory Data Analysis (EDA)
3. Statistical analysis based on the handbook

The EDA and statistics sections mainly use **Pandas, NumPy and Matplotlib**. **Statsmodels is used only for the ADF test


# Section 1: Environment Setup
Install required libraries and configure environment for clean, reproducible output.

In [ ]:
!pip install yfinance -q
!pip install statsmodels -q
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt


# **Section 2: Download and Inspect Data**
Download 5 years of daily OHLCV data and inspect structure.

In [ ]:
import yfinance as yf
import pandas as pd
tickers=['AAPL','MSFT','SPY']
data = yf.download(tickers, period="5y", interval="1d", auto_adjust=True, progress=False)
data.shape

In [ ]:
data.columns = [f"{ticker}_{metric}" for metric, ticker in data.columns]
data=data.reset_index()
data.head()


**Section 2.1: Basic Understanding - Data Structure**

In [ ]:
data.info()

In [ ]:
data.describe()

# **Section 3: Data Cleaning Pipeline**
Execute comprehensive cleaning process.


In [ ]:
data['Date'] = pd.to_datetime(data['Date'])

In [ ]:
dup=data.duplicated().sum()
dup

In [ ]:
price_cols = [col for col in data.columns if any(metric in col for metric in ['Open', 'High', 'Low', 'Close', 'Volume'])]
data[price_cols] = data[price_cols].ffill()
data[price_cols] = data[price_cols].bfill()

Initial inspection (data.isnull().sum()) revealed zero missing values in the raw dataset. However, .ffill() and .bfill() have been retained in the pipeline as a defensive programming measure to ensure the pipeline remains robust against unexpected data gaps in future data downloads.

In [ ]:
data.isnull().sum()

In [ ]:
data

# **Pre-column cleaning notes**



In [ ]:
cleaning_notes = {
    'Date': {
        'Type': 'datetime64[ns]',
        'Issue Found': 'None',
        'Action Taken': 'Parsed with pd.to_datetime()'
    },
    'AAPL_Close / MSFT_Close / SPY_Close': {
        'Type': 'float32',
        'Issue Found': '0 nulls',
        'Action Taken': 'fill/bfill (no-op), cast to float32'
    },
    'AAPL_High / MSFT_High / SPY_High': {
        'Type': 'float32',
        'Issue Found': '0 nulls',
        'Action Taken': 'Price integrity validated, cast to float32'
    },
    'AAPL_Low / MSFT_Low / SPY_Low': {
        'Type': 'float32',
        'Issue Found': '0 nulls',
        'Action Taken': 'Price integrity validated, cast to float32'
    },
    'AAPL_Open / MSFT_Open / SPY_Open': {
        'Type': 'float32',
        'Issue Found': '0 nulls',
        'Action Taken': 'Price integrity validated, cast to float32'
    },
    'AAPL_Volume / MSFT_Volume / SPY_Volume': {
        'Type': 'int32',
        'Issue Found': '0 nulls',
        'Action Taken': 'Cast to int32 (logically whole numbers, yfinance returns float64)'
    }
}

notes_df = pd.DataFrame(cleaning_notes).T
notes_df

# **Section 4: Price Integrity Validation**
What we're checking:
For any given trading day, the relationship between OHLC prices should satisfy: Low ≤ Open, Close ≤ High

Why this matters:
Violations indicate data corruption, entry errors, or outliers that could distort analysis.

Approach:
For each stock, we validate that daily price relationships are logically consistent.

In [ ]:
tickers_in_data = list(set([col.split('_')[0] for col in price_cols]))
price_issues = {}
for ticker in sorted(tickers_in_data):
    open_col = f"{ticker}_Open"
    high_col = f"{ticker}_High"
    low_col = f"{ticker}_Low"
    close_col = f"{ticker}_Close"

    if all(col in data.columns for col in [open_col, high_col, low_col, close_col]):
        issue1 = ((data[low_col] > data[open_col]) | (data[open_col] > data[high_col])).sum()
        issue2 = ((data[low_col] > data[close_col]) | (data[close_col] > data[high_col])).sum()
        issue3 = (data[low_col] > data[high_col]).sum()
        total_issues = issue1 + issue2 + issue3
        price_issues[ticker] = total_issues
        if total_issues == 0:
            print(f"All {len(data)} records satisfy Low ≤ Open,Close ≤ High")
        else:
            print(f"Found {total_issues} price relationship violations")
            if issue1 > 0:
                print(f"Low > Open OR Open > High: {issue1} records")
            if issue2 > 0:
                print(f"Low > Close OR Close > High: {issue2} records")
            if issue3 > 0:
                print(f"Low > High: {issue3} records")

# **Section 5: Memory Optimization**


In [ ]:
memory_before = data.memory_usage(deep=True).sum() / 1024**2


In [ ]:
float_cols = data.select_dtypes(include=['float64']).columns
for col in float_cols:
    data[col] = data[col].astype('float32')

In [ ]:
int_cols = data.select_dtypes(include=['int64']).columns
for col in int_cols:
    if (data[col].min() >= -2**31) and (data[col].max() <= 2**31 - 1):
        data[col] = data[col].astype('int32')

In [ ]:
memory_after = data.memory_usage(deep=True).sum() / 1024**2
memory_saved = memory_before - memory_after
memory_saved_pct = (memory_saved / memory_before) * 100

# **Section 6: Exploratory Data Analysis (EDA)**

According to the handbook, EDA should first answer:

- How many rows and columns are present?
- What does one row represent?
- What is the date range?
- What is the shape of each important numerical column?
- Are the values skewed?
- How do the stocks move over time?

Here, one row represents **one trading day**.


## **6.1 Check the final shape and date range**


In [ ]:
print("Number of rows:", data.shape[0])
print("Number of columns:", data.shape[1])

print("Starting date:", data['Date'].min())
print("Ending date:", data['Date'].max())


## **6.2 Select the closing-price columns**

The closing price is the last traded price recorded for a stock on a trading day.


In [ ]:
close_columns = [
    'AAPL_Close',
    'MSFT_Close',
    'SPY_Close'
]

data[close_columns].head()


## **6.3 Descriptive statistics of closing prices**

`describe()` provides the count, mean, standard deviation, minimum, quartiles and maximum.


In [ ]:
data[close_columns].describe().round(2)


## **6.4 Compare mean and median**

The handbook explains that the mean can be affected by extreme values.  
The median represents the middle observation and is less affected by unusually high or low values.


In [ ]:
closing_price_summary = data[close_columns].agg([
    'mean',
    'median',
    'std',
    'min',
    'max',
    'skew'
])

closing_price_summary.round(3)


### Interpretation

- A positive skew value shows a longer right tail.
- A negative skew value shows a longer left tail.
- When the mean and median are close, the distribution is more balanced.
- Raw stock prices change over time, so they should not be used directly for forecasting statistics.


## **6.5 Plot closing prices over time**


In [ ]:
data.plot(
    x='Date',
    y=close_columns,
    figsize=(12, 6),
    title='Closing Prices of AAPL, MSFT and SPY'
)

plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.show()


The stocks have different price levels. To compare their growth fairly, we can set the first value of every stock equal to 100.


## **6.6 Normalise the closing prices**


In [ ]:
normalised_prices = data[close_columns] / data[close_columns].iloc[0] * 100

normalised_prices['Date'] = data['Date']

normalised_prices.head()


In [ ]:
normalised_prices.plot(
    x='Date',
    y=close_columns,
    figsize=(12, 6),
    title='Normalised Stock Prices: Starting Value = 100'
)

plt.xlabel('Date')
plt.ylabel('Normalised Price')
plt.show()


## **6.7 Trading-volume analysis**

Volume shows how many shares were traded during a day.


In [ ]:
volume_columns = [
    'AAPL_Volume',
    'MSFT_Volume',
    'SPY_Volume'
]

data[volume_columns].describe().round(2)


In [ ]:
data.plot(
    x='Date',
    y='AAPL_Volume',
    figsize=(12, 5),
    title='AAPL Daily Trading Volume'
)

plt.xlabel('Date')
plt.ylabel('Volume')
plt.show()


In [ ]:
data.plot(
    x='Date',
    y='MSFT_Volume',
    figsize=(12, 5),
    title='MSFT Daily Trading Volume'
)

plt.xlabel('Date')
plt.ylabel('Volume')
plt.show()


In [ ]:
data.plot(
    x='Date',
    y='SPY_Volume',
    figsize=(12, 5),
    title='SPY Daily Trading Volume'
)

plt.xlabel('Date')
plt.ylabel('Volume')
plt.show()


# **Section 7: Returns Analysis**

The handbook says that stock prices usually trend over time.  
Therefore, statistical analysis should use **returns instead of raw prices**.

We first calculate simple daily percentage returns for easy interpretation.


## **7.1 Calculate daily percentage returns**


In [ ]:
data['AAPL_Daily_Return'] = data['AAPL_Close'].pct_change()
data['MSFT_Daily_Return'] = data['MSFT_Close'].pct_change()
data['SPY_Daily_Return'] = data['SPY_Close'].pct_change()

daily_return_columns = [
    'AAPL_Daily_Return',
    'MSFT_Daily_Return',
    'SPY_Daily_Return'
]

data[daily_return_columns].head()


The first return is missing because there is no previous trading day available for comparison.


## **7.2 Summary of daily returns**


In [ ]:
daily_return_summary = data[daily_return_columns].agg([
    'mean',
    'median',
    'std',
    'min',
    'max',
    'skew'
])

daily_return_summary.round(5)


### Interpretation

- Mean return shows the average daily movement.
- Median return shows the middle daily movement.
- Standard deviation shows daily volatility.
- A larger standard deviation means greater price variation.
- Skewness shows whether unusually positive or negative returns create a longer tail.


## **7.3 Distribution of AAPL daily returns**


In [ ]:
data['AAPL_Daily_Return'].hist(
    bins=40,
    figsize=(9, 5)
)

plt.title('Distribution of AAPL Daily Returns')
plt.xlabel('Daily Return')
plt.ylabel('Number of Days')
plt.show()


## **7.4 Distribution of MSFT daily returns**


In [ ]:
data['MSFT_Daily_Return'].hist(
    bins=40,
    figsize=(9, 5)
)

plt.title('Distribution of MSFT Daily Returns')
plt.xlabel('Daily Return')
plt.ylabel('Number of Days')
plt.show()


## **7.5 Distribution of SPY daily returns**


In [ ]:
data['SPY_Daily_Return'].hist(
    bins=40,
    figsize=(9, 5)
)

plt.title('Distribution of SPY Daily Returns')
plt.xlabel('Daily Return')
plt.ylabel('Number of Days')
plt.show()


## **7.6 Correlation between daily returns**

Correlation ranges from -1 to +1:

- Near +1: the returns usually move in the same direction.
- Near -1: the returns usually move in opposite directions.
- Near 0: there is no clear linear relationship.

Correlation does **not** prove that one stock causes another stock to move.


In [ ]:
return_correlation = data[daily_return_columns].corr()

return_correlation.round(3)


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    data['SPY_Daily_Return'],
    data['AAPL_Daily_Return'],
    alpha=0.4
)

plt.title('AAPL Return vs SPY Return')
plt.xlabel('SPY Daily Return')
plt.ylabel('AAPL Daily Return')
plt.show()


# **Section 8: Statistical Analysis According to the Handbook**


## **8.1 Calculate log returns**



Log returns are calculated as:

`log(today's closing price / previous day's closing price)`


In [ ]:
data['AAPL_Log_Return'] = np.log(
    data['AAPL_Close'] / data['AAPL_Close'].shift(1)
)

data['MSFT_Log_Return'] = np.log(
    data['MSFT_Close'] / data['MSFT_Close'].shift(1)
)

data['SPY_Log_Return'] = np.log(
    data['SPY_Close'] / data['SPY_Close'].shift(1)
)

log_return_columns = [
    'AAPL_Log_Return',
    'MSFT_Log_Return',
    'SPY_Log_Return'
]

data[log_return_columns].head()


## **8.2 Descriptive statistics of log returns**


In [ ]:
log_return_summary = data[log_return_columns].agg([
    'mean',
    'median',
    'std',
    'min',
    'max',
    'skew'
])

log_return_summary.round(5)


## **8.3 ADF stationarity test**

The Augmented Dickey-Fuller test checks whether a time series is stationary.

### Hypotheses

- **Null hypothesis:** The series is not stationary.
- **Alternative hypothesis:** The series is stationary.

### Decision rule

- If p-value < 0.05, reject the null hypothesis.
- If p-value ≥ 0.05, there is not enough evidence to call the series stationary.

`statsmodels` is used only because the handbook specifically requires the ADF test.


In [ ]:
from statsmodels.tsa.stattools import adfuller


### ADF test for AAPL log returns


In [ ]:
aapl_log_returns = data['AAPL_Log_Return'].dropna()

aapl_adf_result = adfuller(aapl_log_returns)

print("ADF statistic:", round(aapl_adf_result[0], 4))
print("p-value:", aapl_adf_result[1])

if aapl_adf_result[1] < 0.05:
    print("Conclusion: AAPL log returns are stationary.")
else:
    print("Conclusion: AAPL log returns are not stationary.")


### ADF test for MSFT log returns


In [ ]:
msft_log_returns = data['MSFT_Log_Return'].dropna()

msft_adf_result = adfuller(msft_log_returns)

print("ADF statistic:", round(msft_adf_result[0], 4))
print("p-value:", msft_adf_result[1])

if msft_adf_result[1] < 0.05:
    print("Conclusion: MSFT log returns are stationary.")
else:
    print("Conclusion: MSFT log returns are not stationary.")


### ADF test for SPY log returns


In [ ]:
spy_log_returns = data['SPY_Log_Return'].dropna()

spy_adf_result = adfuller(spy_log_returns)

print("ADF statistic:", round(spy_adf_result[0], 4))
print("p-value:", spy_adf_result[1])

if spy_adf_result[1] < 0.05:
    print("Conclusion: SPY log returns are stationary.")
else:
    print("Conclusion: SPY log returns are not stationary.")


## **8.4 95% confidence interval for average daily log return**

A confidence interval provides a reasonable range for the average daily log return instead of reporting only one value.

The calculation below uses Pandas and NumPy.


### AAPL confidence interval


In [ ]:
aapl_mean = aapl_log_returns.mean()
aapl_standard_error = (
    aapl_log_returns.std(ddof=1)
    /
    np.sqrt(len(aapl_log_returns))
)

aapl_lower = aapl_mean - 1.96 * aapl_standard_error
aapl_upper = aapl_mean + 1.96 * aapl_standard_error

print("Average daily log return:", round(aapl_mean, 6))
print("95% confidence interval:",
      round(aapl_lower, 6),
      "to",
      round(aapl_upper, 6))


### MSFT confidence interval


In [ ]:
msft_mean = msft_log_returns.mean()
msft_standard_error = (
    msft_log_returns.std(ddof=1)
    /
    np.sqrt(len(msft_log_returns))
)

msft_lower = msft_mean - 1.96 * msft_standard_error
msft_upper = msft_mean + 1.96 * msft_standard_error

print("Average daily log return:", round(msft_mean, 6))
print("95% confidence interval:",
      round(msft_lower, 6),
      "to",
      round(msft_upper, 6))


### SPY confidence interval


In [ ]:
spy_mean = spy_log_returns.mean()
spy_standard_error = (
    spy_log_returns.std(ddof=1)
    /
    np.sqrt(len(spy_log_returns))
)

spy_lower = spy_mean - 1.96 * spy_standard_error
spy_upper = spy_mean + 1.96 * spy_standard_error

print("Average daily log return:", round(spy_mean, 6))
print("95% confidence interval:",
      round(spy_lower, 6),
      "to",
      round(spy_upper, 6))


## **8.5 Split the data by time**


The first 80% of rows are used as the past/training period.  
The final 20% are used as the future/test period.


In [ ]:
split_position = int(len(data) * 0.80)

train_data = data.iloc[:split_position].copy()
test_data = data.iloc[split_position:].copy()

print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

print("Training period:",
      train_data['Date'].min(),
      "to",
      train_data['Date'].max())

print("Testing period:",
      test_data['Date'].min(),
      "to",
      test_data['Date'].max())


## **8.6 Create the naive baseline**



**Tomorrow's price = today's price**

The previous day's closing price becomes the naive prediction for the next day.


In [ ]:
data['AAPL_Naive_Prediction'] = data['AAPL_Close'].shift(1)
data['MSFT_Naive_Prediction'] = data['MSFT_Close'].shift(1)
data['SPY_Naive_Prediction'] = data['SPY_Close'].shift(1)

test_data = data.iloc[split_position:].copy()

test_data[[
    'Date',
    'AAPL_Close',
    'AAPL_Naive_Prediction'
]].head()


## **8.7 Calculate RMSE and MAPE using NumPy**

- **RMSE** shows prediction error in price units.
- **MAPE** shows average prediction error as a percentage.
- Smaller values are better.


### AAPL naive-baseline score


In [ ]:
aapl_actual = test_data['AAPL_Close']
aapl_prediction = test_data['AAPL_Naive_Prediction']

aapl_rmse = np.sqrt(
    np.mean(
        (aapl_actual - aapl_prediction) ** 2
    )
)

aapl_mape = np.mean(
    np.abs(
        (aapl_actual - aapl_prediction)
        /
        aapl_actual
    )
) * 100

print("AAPL RMSE:", round(aapl_rmse, 3))
print("AAPL MAPE:", round(aapl_mape, 3), "%")


### MSFT naive-baseline score


In [ ]:
msft_actual = test_data['MSFT_Close']
msft_prediction = test_data['MSFT_Naive_Prediction']

msft_rmse = np.sqrt(
    np.mean(
        (msft_actual - msft_prediction) ** 2
    )
)

msft_mape = np.mean(
    np.abs(
        (msft_actual - msft_prediction)
        /
        msft_actual
    )
) * 100

print("MSFT RMSE:", round(msft_rmse, 3))
print("MSFT MAPE:", round(msft_mape, 3), "%")


### SPY naive-baseline score


In [ ]:
spy_actual = test_data['SPY_Close']
spy_prediction = test_data['SPY_Naive_Prediction']

spy_rmse = np.sqrt(
    np.mean(
        (spy_actual - spy_prediction) ** 2
    )
)

spy_mape = np.mean(
    np.abs(
        (spy_actual - spy_prediction)
        /
        spy_actual
    )
) * 100

print("SPY RMSE:", round(spy_rmse, 3))
print("SPY MAPE:", round(spy_mape, 3), "%")


## **8.8 Put the baseline results in one table**


In [ ]:
baseline_results = pd.DataFrame({
    'Stock': ['AAPL', 'MSFT', 'SPY'],
    'RMSE': [aapl_rmse, msft_rmse, spy_rmse],
    'MAPE_Percent': [aapl_mape, msft_mape, spy_mape]
})

baseline_results.round(3)


## **8.9 Plot actual and naive AAPL prices**


In [ ]:
test_data.plot(
    x='Date',
    y=['AAPL_Close', 'AAPL_Naive_Prediction'],
    figsize=(12, 6),
    title='AAPL Actual Price vs Naive Prediction'
)

plt.xlabel('Date')
plt.ylabel('Price')
plt.show()


## **8.10 Plot actual and naive MSFT prices**


In [ ]:
test_data.plot(
    x='Date',
    y=['MSFT_Close', 'MSFT_Naive_Prediction'],
    figsize=(12, 6),
    title='MSFT Actual Price vs Naive Prediction'
)

plt.xlabel('Date')
plt.ylabel('Price')
plt.show()


## **8.11 Plot actual and naive SPY prices**


In [ ]:
test_data.plot(
    x='Date',
    y=['SPY_Close', 'SPY_Naive_Prediction'],
    figsize=(12, 6),
    title='SPY Actual Price vs Naive Prediction'
)

plt.xlabel('Date')
plt.ylabel('Price')
plt.show()


# **Final Interpretation**

After running the notebook, write the conclusion using the output values:

1. The dataset contains one row for each trading day.
2. Raw prices show a time trend, so they are not suitable for direct statistical forecasting.
3. Daily returns and log returns fluctuate around a much more stable level.
4. The ADF p-values show whether the log-return series are stationary.
5. Return correlations show how AAPL, MSFT and SPY move together, but correlation does not prove causation.
6. The time split keeps future observations out of the training period and prevents leakage.
7. The naive forecast provides the minimum benchmark that a future ARIMA or Prophet model must beat.
8. RMSE and MAPE are used because stock-price forecasting is a numerical prediction problem, not a classification problem.


# **Important Limitations**

- The data is downloaded live, so exact results can change depending on the date the notebook is run.
- Historical relationships do not guarantee future stock performance.
- Correlation does not prove that one stock causes another to move.
- A low forecasting error does not guarantee profitable trading after fees and market changes.
- The notebook creates a baseline only; it does not claim that the naive method is a final trading model.


# **Section 9: Validation and Export**
Validate the complete cleaned and analysed dataset, then export it to CSV.


In [ ]:
def validate(df, filename="clean_stock_data.csv"):
  print("starting to validate the data.")
  missing_vals=df.isnull().sum().sum()
  if missing_vals>0:
    print(f"Validation failed: found {missing_vals} missing values")
    return False
  duplicate_rows=df.duplicated().sum()
  if duplicate_rows>0:
    print(f"Validation failed: found {duplicate_rows} duplicate rows")
    return False
  price_cols=[col for col in df.columns if 'Close' in col or 'Open' in col or 'High' in col or 'Volume' in col or 'Low' in col]
  for col in price_cols:
    if(df[col]<=0).any():
      print(f"Validation failed: column {col} contains negative or zero prices.")
      return False # Added return False here
  if 'index' in df.columns:
        print(" Validation Failed: The unwanted 'index' column is still present.")
        return False
  print("All validations passed! Data is clean.")
  df.to_csv(filename, index=False)
  print(f"File successfully saved as '{filename}'")
  return True
validate(data)

In [ ]:
import os
output_filename = "clean_stock_data.csv"
data.to_csv(output_filename, index=False)

file_size = os.path.getsize(output_filename) / 1024